# About

This notebook will be for the text detection and segmentation of the notes. That is, not the conversion of handwriting to text, but the separation of handwriting into different lines/categories

**Vision**

What it does

- Utilize YoloV11n-seg (most recent smallest segmentation model that Ultralytics offers)
- Separates each line into a different segmentation
- Multiple categories for segmentation, corresponding to different stylings

I/O

- Pass in the raw scanned image of a page (maybe segmented into smaller portions already, cropped a little)
- Segments each line tightly, with a certain label
    - Could be something like 'header', 'main content', 'list point', 'image'
- Outputs segmentations, which are then individually converted to text and reassembled based on classes and location

Training Data

- Will (most likely) be using my Hands on Machine Learning notes to start, but may also do other class notes, or just some notes that seem like they have good formatting that I like
- Will have to manually segment pages, as well as translate them, whcih may take a little while

Possible Hurdles

- Numerical things that would be good in latex might be hard to do, could end up translating them but I'm not sure of the TrOCR model works that way (since it's trying to match visually as well)
- There might be a lot of training data needed in order to really tune this well, as well as a variety (different types of notebooks, pens, writing clarity, etc)
- It'll probably take a while to compute, will have to figure out what to do with that (maybe combine into one line? multiple lines in one request?)


# Imports

In [1]:
from ultralytics import YOLO

# First Approach

We go line by line, putting bounding boxes around each line. We'll have five classes:

1. Header 
2. Text
3. Unordered Bullet
4. Ordered Bullet
5. Figure

This way, we're able to classify the different types of text. The idea for heirarchy is this:

- Have labelling for bounding boxes be very precise when it comes to alignment
- Use the left-edge of the bounding box to group different sections 
- Use the space between bounding boxes to separate paragraphs, figure out line breaks

## Tiny Test

Only three images labelled by tonight, but I want to see if thats even close to enough. I fell like it might be not too bad, since there's so many bounding boxes for each.

In [2]:
# Load the models
nano_model = YOLO('yolo11n-obb.pt')
small_model = YOLO('yolo11s-obb.pt')
medium_model = YOLO('yolo11m-obb.pt')

In [3]:
# Define function for training, validation
def train_model(model, yaml_path):
    result = model.train(
        data=yaml_path,
        epochs=10,
        imgsz=1024
    )

    return result.results_dict

In [4]:
model_results = []
yaml_path = r'dataset\tiny_data.yaml'

for model in [nano_model, small_model, medium_model]:
    results = train_model(model, yaml_path)
    model_results.append(results)

Ultralytics 8.3.174  Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6141MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\tiny_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True

train: Scanning C:\Users\gjbur\Desktop\Code\note-transfer\development\dataset\labels... 2 images, 0 backgrounds, 0 corrupt: 100%|██████████| 2/2 [00:00<00:00, 192.63it/s]

train: New cache created: C:\Users\gjbur\Desktop\Code\note-transfer\development\dataset\labels.cache


val: Fast image access  (ping: 0.40.0 ms, read: 22.10.0 MB/s, size: 186.5 KB)


val: Scanning C:\Users\gjbur\Desktop\Code\note-transfer\development\dataset\labels... 1 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1/1 [00:00<00:00, 77.35it/s]

...

      10/10      3.21G      4.489      3.826      4.791         60       1024: 100%|██████████| 1/1 [00:00<00:00,  3.25it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 10.79it/s]

                   all          1         31          0          0          0          0



10 epochs completed in 0.007 hours.
Optimizer stripped from runs\obb\train9\weights\last.pt, 42.3MB
Optimizer stripped from runs\obb\train9\weights\best.pt, 42.3MB

Validating runs\obb\train9\weights\best.pt...
Ultralytics 8.3.174  Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6141MiB)
YOLO11m-obb summary (fused): 134 layers, 20,882,338 parameters, 0 gradients, 71.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 12.10it/s]


                   all          1         31          0          0          0          0
                Header          1          2          0          0          0          0
      Unordered Bullet          1          4          0          0          0          0
        Ordered Bullet          1         20          0          0          0          0
                  Text          1          5          0          0          0          0
Speed: 1.5ms preprocess, 76.8ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs\obb\train9


In [5]:
model_results

[{'metrics/precision(B)': np.float64(0.0),
  'metrics/recall(B)': np.float64(0.0),
  'metrics/mAP50(B)': np.float64(0.0),
  'metrics/mAP50-95(B)': np.float64(0.0),
  'fitness': np.float64(0.0)},
 {'metrics/precision(B)': np.float64(0.0),
  'metrics/recall(B)': np.float64(0.0),
  'metrics/mAP50(B)': np.float64(0.0),
  'metrics/mAP50-95(B)': np.float64(0.0),
  'fitness': np.float64(0.0)},
 {'metrics/precision(B)': np.float64(0.0),
  'metrics/recall(B)': np.float64(0.0),
  'metrics/mAP50(B)': np.float64(0.0),
  'metrics/mAP50-95(B)': np.float64(0.0),
  'fitness': np.float64(0.0)}]

Yeah, no. We'll probably need to do **a lot** more labelling.